In [1]:
%display latex

# Devoir 01 - Calcul Formel
## Arithmétique Multiprécision en SageMath

**Nom :** Diaw Papa Amadou  
**Cours :** Calcul Formel - M1 Algébre Appliquée && Cryptographie  
**Date :** 2025-2026

In [2]:
# longueur maximale
mylambdamax=5

# élément de ZZ
def myentier(xA):
    try:
        x=ZZ(xA)
    except TypeError:
        raise TypeError("ERREUR : N'est pas un entier",xA)
    return x

In [3]:
myentier(4)

4

In [4]:
# Test cas normaux
print(myentier(5))
print(myentier(-3))
print(myentier(0))

# Test cas extrêmes
print(myentier(100000))


5
-3
0
100000


## Fonction 2 : myentier_entmp
### Conversion entier → entier multiprécision

In [5]:
# conversion entier → entier multiprécision
def myentier_entmp(nA, betaA):
    n = myentier(nA)
    beta = myentier(betaA)
    if beta < 2:# en base toujours superieure ou egal 2
        raise TypeError("ERREUR : Base < 2", beta)
    if n == 0:
        return [0]
    sigma = ZZ((1 - sign(n)) / 2)
    entmp = [sigma]# sigma represente le signe de n ,egal à 0 si'il est positif et 1 sinon
    q = abs(n)
    longueur = 0
    while q > 0:
        longueur += 1
        if longueur > mylambdamax:
            raise TypeError("ERREUR : Débordement longueur", longueur, '>', mylambdamax)
        q, r = q.quo_rem(beta)
        entmp.append(r)
    return entmp

### Tests de myentier_entmp

In [6]:
# Test cas normaux
print(myentier_entmp(123, 10))   # base 10
print(myentier_entmp(-123, 10))  # négatif
print(myentier_entmp(0, 10))     # zéro
print(myentier_entmp(5, 2))      # base 2 (binaire)

# Test cas extrêmes
print(myentier_entmp(1, 10))     # plus petit positif
print(myentier_entmp(-1, 10))    # plus petit négatif

[0, 3, 2, 1]
[1, 3, 2, 1]
[0]
[0, 1, 0, 1]
[0, 1]
[1, 1]


## Fonction 3 : myentmpcanonique
### Entier multiprécision : forme canonique

In [7]:
# entier multiprécision : forme canonique
def myentmpcanonique(entmpA, betaA):
    beta = myentier(betaA)
    if beta < 2:
        raise TypeError("ERREUR : Base < 2", beta)
    longueur = len(entmpA) - 1 # Longueur de la representation
    if longueur < 0:
        raise TypeError("ERREUR : Liste vide", entmpA)
    if longueur > mylambdamax:
        raise TypeError("ERREUR : Débordement longueur", longueur, '>', mylambdamax)
    sigma = myentier(entmpA[0])
    if not(sigma in [0,1]): # Verification de la signe 
        raise TypeError("ERREUR : Mauvais sigma", sigma)
    imax = longueur
    go = True # go est un interrupteur
    for i in range(longueur, 0, -1): # on parcours la liste de la droite vers la gauche pour trouver le dernier 0
        chiffre = entmpA[i]
        if not(0 <= chiffre < beta):
            raise TypeError("ERREUR : Mauvais chiffre", chiffre)
        if (go and chiffre != 0): # si on tombe sur 0 on l'enleve et on descend 
            imax = i
            go = False
    if go:# si la condition est vrai et que tous les valeurs sont nul tous les chiffres étaient 0
        return [0]
    res = entmpA[0:imax+1]
    if len(res) == 1:
        return [0]
    return res

### Tests de myentmpcanonique

In [8]:
# Test cas normaux - suppression des zéros inutiles
print(myentmpcanonique([0, 3, 2, 1, 0, 0], 10))  # doit donner [0, 3, 2, 1]
print(myentmpcanonique([0, 5], 10))                # déjà canonique
print(myentmpcanonique([1, 3, 2, 1], 10))          # négatif

# Test cas extrêmes
print(myentmpcanonique([0], 10))                   # zéro
print(myentmpcanonique([0, 0, 0, 0], 10))          # que des zéros → [0]
print(myentmpcanonique([0, 1], 2))              # base 2

[0, 3, 2, 1]
[0, 5]
[1, 3, 2, 1]
[0]
[0]
[0, 1]


## Fonction 4 : mylongueur
### Longueur d'un entier ou d'un entier multiprécision

In [9]:
# longueur d'un entier ou d'un entier multiprécision
def mylongueur(xA, betaA):
    if xA in ZZ:# longueur d'un entier
        if betaA in ZZ and betaA >= 2:
            longueur = 1 + floor(log(abs(xA), betaA))
        else:
            raise TypeError("ERREUR : Mauvais argument(s)", xA, betaA)
    else: # longueur d'un entier multiprécision
        entmp = myentmpcanonique(xA, betaA)
        longueur = len(entmp) - 1
    return longueur

In [10]:
# Test avec des entiers normaux
print(mylongueur(123, 10))    # 3 chiffres en base 10
print(mylongueur(8, 2))       # 1000 en base 2 → 4 chiffres
print(mylongueur(1, 10))      # 1 chiffre

# Test avec des entiers multiprécision
print(mylongueur([0, 3, 2, 1], 10))   # longueur 3
print(mylongueur([0, 1], 2))           # longueur 1

# Test cas extrêmes
print(mylongueur([0, 0], 2))
  

3
4
1
3
1
0


## Fonction 5 : myentmp_entier
### Conversion entier multiprécision → entier

In [11]:
# conversion entier multiprécision → entier
def myentmp_entier(entmpA, betaA):
    beta = myentier(betaA)
    entmp = myentmpcanonique(entmpA, beta)
    sigma = entmp[0]
    longueur = mylongueur(entmp, beta)
    n = 0
    p = 1
    for i in [1..longueur]:
        chiffre = myentier(entmpA[i])
        if not(0 <= chiffre < beta):
            raise TypeError("ERREUR : Mauvais chiffre", chiffre)
        n += p * chiffre
        p *= beta
    n = (-1)^sigma * n
    return n

### Tests de myentmp_entier

In [12]:
# Tests cas normaux - conversion multiprécision → entier
print(myentmp_entier([0, 3, 2, 1], 10))   # doit donner 123
print(myentmp_entier([1, 3, 2, 1], 10))   # doit donner -123
print(myentmp_entier([0], 10))             # doit donner 0
print(myentmp_entier([0, 1], 10))          # doit donner 1
print(myentmp_entier([0, 1], 2))           # base 2 → doit donner 1

# Test cas extrêmes
print(myentmp_entier([1, 1], 10))          # doit donner -1
print(myentmp_entier([0, 0, 0, 1], 10))   # doit donner 100

123
-123
0
1
1
-1
100


## Fonction 6 : myentmpsmemelongueur
### Mettre des entiers multiprécision à la même longueur

In [13]:
# entiers multiprécision : même longueur
def myentmpsmemelongueur(*args):
    if len(args) == 0:
        raise TypeError("ERREUR : Manque beta", args)
    if len(args) == 1:
        return ()
    beta = myentier(args[-1])# Beta est le dernier element donné en entrée
    entmps = args[:-1] # On prend les restes excepté beta 
    res = tuple(myentmpcanonique(e, beta) for e in entmps)
    m = max(len(e) for e in res) # on prend la plus grande liste 
    res = tuple(e + [0]*(m - len(e)) for e in res)# C'est la qu'on complete les autres listes pour avoir la méme longueur que la longueur max
    return res

### Tests de myentmpsmemelongueur

In [14]:
# Tests cas normaux - mise à même longueur
print(myentmpsmemelongueur([0, 1], [0, 3, 2], 10))      # longueurs différentes
print(myentmpsmemelongueur([0, 1], [0, 1], 10))          # même longueur
print(myentmpsmemelongueur([0, 1], [0, 3, 2], [1, 1], 10))  # 3 entiers

# Tests cas extrêmes
print(myentmpsmemelongueur(10))          # un seul argument → retourne ()
print(myentmpsmemelongueur([0], [0], 10))  # deux zéros

([0, 1, 0], [0, 3, 2])
([0, 1], [0, 1])
([0, 1, 0], [0, 3, 2], [1, 1, 0])
()
([0], [0])


## Fonction 7 : myADD
### Simulation de l'instruction ADD du processeur

In [15]:
# instruction ADD du processeur : simulation 
def myADD(aA,bA,rhoA,betaA):
    a=myentier(aA)
    b=myentier(bA)
    beta=myentier(betaA)
    rho=myentier(rhoA)
    if not(beta>=2 and 0<=a<beta and 0<=b<beta and rho in [0,1]):
        raise TypeError("ERREUR : Mauvais argument(s)",a,b,rho,beta)
    c,rhoprime=a+b+rho,0
    if c>=beta:
        c,rhoprime=c-beta,1
    return c,rhoprime

### Tests de myADD

In [16]:
# Tests cas normaux
print(myADD(3, 4, 0, 10))    # 3+4+0 = 7, pas de retenue → (7, 0)
print(myADD(7, 8, 0, 10))    # 7+8+0 = 15 → (5, 1) retenue
print(myADD(7, 8, 1, 10))    # 7+8+1 = 16 → (6, 1) retenue
print(myADD(0, 0, 0, 10))    # 0+0+0 = 0 → (0, 0)

# Tests base 2
print(myADD(1, 1, 0, 2))     # 1+1+0 = 2 → (0, 1) retenue
print(myADD(1, 1, 1, 2))     # 1+1+1 = 3 → (1, 1) retenue

# Tests cas extrêmes
print(myADD(9, 9, 1, 10))    # 9+9+1 = 19 → (9, 1)
print(myADD(0, 0, 1, 2))     # 0+0+1 = 1 → (1, 0)

(7, 0)
(5, 1)
(6, 1)
(0, 0)
(0, 1)
(1, 1)
(9, 1)
(1, 0)


## Fonction 8 : myalgo321
### Algorithme 3.2.1 - Addition de deux entiers multiprécision

In [17]:
# Algorithme 3.2.1
def myalgo321(xA,yA,betaA):
    beta=myentier(betaA)
    x,y=myentmpsmemelongueur(xA,yA,beta)
    longueur=len(x)-1
    s=[0]
    rho=0
    for i in [1..longueur]:
        si,rho=myADD(x[i],y[i],rho,beta)
        s.append(si)
    if rho>0:
        if longueur >= mylambdamax:
            raise TypeError("ERREUR : Débordement longueur",longueur,'>',mylambdamax)
        else:
            sn=rho
            s.append(sn)
    return s

### Tests de myalgo321

In [18]:
# Tests cas normaux - addition multiprécision
print(myalgo321([0, 3, 2, 1], [0, 4, 5, 1], 10))  # 123 + 154 = 277
print(myalgo321([0, 1], [0, 2], 10))                # 1 + 2 = 3
print(myalgo321([0], [0], 10))                       # 0 + 0 = 0

# Tests avec retenue
print(myalgo321([0, 9, 9], [0, 1], 10))             # 99 + 1 = 100
print(myalgo321([0, 5], [0, 5], 10))                 # 5 + 5 = 10

# Test base 2
print(myalgo321([0, 1, 1], [0, 1], 2))              # 3 + 1 = 4

[0, 7, 7, 2]
[0, 3]
[0]
[0, 0, 0, 1]
[0, 0, 1]
[0, 0, 0, 1]


## Fonction 8 : myMUL
### Simulation de l'instruction MUL du processeur

In [19]:
# instruction MUL du processeur : simulation
def myMUL(aA, bA, betaA):
    a = myentier(aA)
    b = myentier(bA)
    beta = myentier(betaA)
    if not(beta >= 2 and 0 <= a < beta and 0 <= b < beta):
        raise TypeError("ERREUR : Mauvais argument(s)", a, b, beta)
    c1, c0 = (a*b).quo_rem(beta)
    return (c0, c1)

### Tests de myMUL

In [20]:
# Tests cas normaux
print(myMUL(3, 4, 10))    # 3*4 = 12 → (2, 1)
print(myMUL(2, 5, 10))    # 2*5 = 10 → (0, 1)
print(myMUL(0, 9, 10))    # 0*9 = 0  → (0, 0)
print(myMUL(1, 1, 10))    # 1*1 = 1  → (1, 0)

# Tests base 2
print(myMUL(1, 1, 2))     # 1*1 = 1  → (1, 0)
print(myMUL(1, 0, 2))     # 1*0 = 0  → (0, 0)

# Tests cas extrêmes
print(myMUL(9, 9, 10))    # 9*9 = 81 → (1, 8)
print(myMUL(0, 0, 10))    # 0*0 = 0  → (0, 0)

(2, 1)
(0, 1)
(0, 0)
(1, 0)
(1, 0)
(0, 0)
(1, 8)
(0, 0)


## Fonction 10 : myalgo331
### Algorithme 3.3.1 - Multiplication d'un entier multiprécision par un chiffre

In [21]:
# Algorithme 3.3.1
def myalgo331(aA,xA,betaA):
    a=myentier(aA)
    beta=myentier(betaA)
    x=myentmpcanonique(xA,beta)
    sigma=x[0]
    n=mylongueur(x,beta)
    P=[0]
    if n==0:
        return P
    p0, d0 = myMUL(a, x[1], beta)
    P.append(p0)
    for i in [1..n-1]:
        c1 , d1 = myMUL(a, x[i+1], beta)
        p1 , rho1 = myADD( c1, d0 ,0,beta)
        d0 , rho0 = myADD( d1, 0 ,rho1,beta)
        P=P+[p1]
    P=myentmpcanonique((P+[d0]),beta)
    return P

### Tests de myalgo331

In [22]:
# Tests cas normaux
print(myalgo331(2, [0, 0, 0, 1], 10))   # 2 * 100 = 200
print(myalgo331(3, [0, 1, 2], 10))       # 3 * 21 = 63
print(myalgo331(0, [0, 1, 2], 10))       # 0 * 21 = 0
print(myalgo331(1, [0, 5, 3], 10))       # 1 * 35 = 35

# Tests base 2
print(myalgo331(1, [0, 1, 1], 2))        # 1 * 3 = 3

# Tests cas extrêmes
print(myalgo331(9, [0, 9, 9], 10))       # 9 * 99 = 891
print(myalgo331(2, [0], 10))             # 2 * 0 = 0

[0, 0, 0, 2]
[0, 3, 6]
[0]
[0, 5, 3]
[0, 1, 1]
[0, 1, 9, 8]
[0]


## Fonction 11 : myalgo333
### Multiplication naïve de 2 entiers multiprécision positifs ou nuls

In [23]:
# Algorithme 3.3.3
def myalgo333(xA, yA, betaA):
    beta = myentier(betaA)
    x, y = myentmpsmemelongueur(xA, yA, beta)
    m = mylongueur(x, beta)
    
    # w0 ← 0
    w = [0]
    
    # FOR i FROM 0 TO m-1
    p_chiffres = []
    for i in [0..m-1]:
        #zi ← xi * y (algorithme 3.3.1)
        z = myalgo331(x[i+1], y, beta)
        
        # si ← wi + zi (algorithme 3.2.1)
        s = myalgo321(w, z, beta)
        
        # pi + w_{i+1}*beta ← si
        p_chiffres.append(s[1] if len(s) > 1 else 0)
        w = [0] + s[2:]  # le reste devient le nouveau w
    
    # construire p final
    p = [0] + p_chiffres + w[1:]
    p = myentmpcanonique(p, beta)
    
    return p

### Tests de myalgo333

In [24]:
print(myalgo333([0, 3, 2], [0, 1, 2], 10))  # 23 * 21 = 483
print(myalgo333([0, 2], [0, 3], 10))          # 2 * 3 = 6
print(myalgo333([0], [0, 5], 10))              # 0 * 5 = 0

[0, 3, 8, 4]
[0, 6]
[0]


### Algorithme 3.3.6 : Multiplication coûteuse de 2 entiers multiprécision positifs ou nuls

In [25]:
# Algorithme 3.3.6
def myalgo336(xA,yA,betaA):
    beta=myentier(betaA)
    x,y=myentmpsmemelongueur(xA,yA,beta)
    y0=myentmp_entier(y, beta)
    p=[0]
    for i in [1..y0]:
        p=myalgo321(p,x,beta)
    return p

### Test pour cet avec les memes valeurs 

In [26]:
print(myalgo336([0, 3, 2], [0, 1, 2], 10))  # 23 * 21 = 483
print(myalgo336([0, 2], [0, 3], 10))          # 2 * 3 = 6
print(myalgo336([0,5], [0], 10))              # 0 * 5 = 0

[0, 3, 8, 4]
[0, 6]
[0]


### Algorithme3.3.8 :Karatsuba(𝑘,𝑥,𝑦),diviser pour mieux regner

In [27]:
# Algorithme 3.3.8
def myalgo338(kA,xA,yA,betaA):
    k=myentier(kA)
    beta=myentier(betaA)
    x,y=myentmpsmemelongueur(xA,yA,beta)
    n=mylongueur(x,beta)
    p=[0]
    if k==0:
        c0, c1 = myMUL(x[1], y[1], beta)
        p = [0, c0]
        if c1 > 0:
            p.append(c1)
        return p
    m = n//2
    a0 = [0] + x[1:m+1]
    a1 = [0] + x[m+1:]
    b0 = [0] + y[1:m+1]
    b1 = [0] + y[m+1:]
    A=myalgo338(k-1,a0,b0,beta)
    B=myalgo338(k-1,a1,b1,beta)
    #a+a'*beta^m=a0+a1
    # Additionner a0 + a1
    s_a = myalgo321( a0 ,a1 ,beta ) 
    # Extraire a et a'
    a = myentmpcanonique([0] + s_a[1:m+1] ,beta)
    a_prime = s_a[m+1] if len(s_a) > m+1 else 0
    #b+b'*beta^m=b0+b1
    # Additionner b0 + b1
    s_b = myalgo321( b0 , b1 , beta )
    # Extraire b et b'
    b = myentmpcanonique([0] + s_b[1:m+1] ,beta)
    b_prime = s_b[m+1] if len(s_b) > m+1 else 0
    # Effectuer appel recursif de karatsuba
    C = myalgo338(k-1,a,b,beta)
    d = myalgo336(a,myentier_entmp(b_prime,beta),beta)
    e = myalgo336(b,myentier_entmp(a_prime,beta),beta)
    f = myalgo321( d , e , beta )
    g = myalgo321( f , B , beta )
    g_decale = [0] + [0]*2*m + g[1:]  # décaler g de 2m positions
    h = myalgo321(A, g_decale, beta)
    j0 = myentmp_entier(C, beta) - myentmp_entier(A, beta) - myentmp_entier(B, beta)
    j = myentier_entmp(j0, beta)
    #calcul de r avec myMUL qui donne deux chiffres r0 et r1
    r0,r1 = myMUL(a_prime,b_prime,beta)
    j_decale = [0] + [0]*m + j[1:]
    r_decale = [0] + [0]*3*m + [r0,r1]
    p = myalgo321(myalgo321(h, j_decale, beta), r_decale, beta)
    return p

### Tests de karatsuba avec les memes valeurs que précédent 

In [28]:
print(myalgo338(1,[0, 3, 2], [0, 1, 2], 10))  # 23 * 21 = 483
print(myalgo338(0,[0, 2], [0, 3], 10))          # 2 * 3 = 6
print(myalgo338(0,[0], [0, 5], 10))              # 0 * 5 = 0

[0, 3, 8, 4]
[0, 6]
[0, 0]


### Algorithme 3.4.1 : Exponentiation naïve

In [29]:
# Algorithme 3.4.1
def myalgo341(aA,kA):
    a=myentier(aA)
    k=myentier(kA)
    p=1
    for i in [1..k]:
        p=p*a
    return p

### Algorithme 3.4.3 Représentation en base 2 d’un entier

In [30]:
def myalgo343(kA):
    k=myentier(kA)
    if k==0:
        return[0]
    p=[]
    while k !=0:
        q ,r= (k).quo_rem(2)
        p.append(r)
        k=q
    return p

    
    

In [31]:
myalgo343(2)

[0, 1]

### Algorithme 3.4.5 : Exponentiation rapide, “Square & Multiply”

In [32]:
# Algorithme 3.4.5
def myalgo341(aA,kA):
    a=myentier(aA)
    k=myentier(kA)
    p=1
    b=a
    while k >0 :
        q,r = (k).quo_rem(2)
        if r==1:
            p=p*b
        k=q
        b=b*b
    return p

In [33]:
myalgo341(2,3)

8